In [9]:
import os
import pandas as pd
import logging
from datetime import datetime
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go
from google.colab import drive

# Mount Google Drive
try:
    drive.mount('/content/drive', force_remount=True)
except Exception as e:
    print(f"Drive mounting failed: {e}")

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger()

class Config:
    def __init__(self):
        base_path = '/content/drive/My Drive'
        if not os.path.exists(base_path):
            raise RuntimeError(f"Google Drive not properly mounted. Base path {base_path} not found.")

        self.FOLDER_PATH = os.path.join(base_path, 'Factordata')
        if not os.path.exists(self.FOLDER_PATH):
            os.makedirs(self.FOLDER_PATH)

        self.PREDICTOR_PATH = os.path.join(base_path, 'seco')
        if not os.path.exists(self.PREDICTOR_PATH):
            os.makedirs(self.PREDICTOR_PATH)

        self.WINDOWS = [3, 6, 12, 24]  # Quarterly, semester, annual, 2-year
        self.RANDOM_STATE = 42
        self.TEST_SIZE = 0.3
        self.N_ESTIMATORS = 100
        self.FACTOR_COLUMNS = [
            'Alpha..annualisiert.',
            'Value.Growth',
            'Small.Large',
            'Momentum',
            'Volatility'
        ]
        self.OUTPUT_DIR = os.path.join(self.FOLDER_PATH, 'analysis_output')
        os.makedirs(self.OUTPUT_DIR, exist_ok=True)

class DataPreprocessor:
    def __init__(self, config):
        self.config = config
        self.scaler = StandardScaler()

    def load_and_preprocess(self, file_path):
        try:
            logger.info(f"Loading file: {os.path.basename(file_path)}")

            df = pd.read_excel(file_path)
            df.columns = df.columns.str.lower()
            if 'date' not in df.columns:
                raise ValueError("'date' column missing")

            df['date'] = pd.to_datetime(df['date'], errors='coerce')
            df = df.dropna(subset=['date']).set_index('date')

            features, targets = [], []

            for col in self.config.FACTOR_COLUMNS:
                col_lower = col.lower()
                if col_lower not in df.columns:
                    logger.warning(f"Column {col} missing in {os.path.basename(file_path)}")
                    continue

                series = pd.to_numeric(df[col_lower], errors='coerce').ffill()
                features.append(series)

                for window in self.config.WINDOWS:
                    rolling_mean = series.rolling(window=window).mean()
                    rolling_std = series.rolling(window=window).std()
                    outliers = ((series - rolling_mean).abs() > 2 * rolling_std).astype(int)
                    targets.append(outliers)

            X = pd.concat(features, axis=1)
            X.columns = self.config.FACTOR_COLUMNS

            y = pd.concat(targets, axis=1)
            y.columns = [f"{col}_w{window}" for col in self.config.FACTOR_COLUMNS
                        for window in self.config.WINDOWS]

            logger.info(f"Processed file {file_path}: X shape {X.shape}, y shape {y.shape}")
            return X, y

        except Exception as e:
            logger.error(f"Error processing {file_path}: {e}")
            return None, None

    def load_additional_features(self, dates_to_align):
        try:
            logger.info("Loading additional features from the 'seco' folder...")

            additional_features = pd.DataFrame()

            for file_name, value_col in [('ks_q.csv', 'ks_q_value'),
                                       ('ks_q_hist.csv', 'ks_q_hist_value')]:
                file_path = os.path.join(self.config.PREDICTOR_PATH, file_name)
                if os.path.exists(file_path):
                    df = pd.read_csv(file_path)
                    df.columns = df.columns.str.lower()
                    if 'date' in df.columns:
                        df['date'] = pd.to_datetime(df['date'], errors='coerce')
                        df = df.dropna(subset=['date']).set_index('date')
                        df = df[df.index.isin(dates_to_align)]
                        df.rename(columns={'value': value_col}, inplace=True)
                        additional_features = pd.concat([additional_features, df], axis=1)
                    else:
                        logger.error(f"'date' column missing in {file_name}.")

            additional_features = additional_features.reset_index().drop_duplicates(
                subset='date').set_index('date')
            logger.info(f"Loaded additional features with shape: {additional_features.shape}")
            return additional_features

        except Exception as e:
            logger.error(f"Error loading additional features: {e}")
            return None

class MultiOutputModelTrainer:
    def __init__(self, config):
        self.config = config
        self.model = None
        self.feature_importance = {}

    def prepare_data(self, X, y):
        # Handle NaN values in features and target
        X = X.fillna(X.mean())
        y = y.fillna(0)  # Fill NaN in target with 0 (no anomaly)

        # Split the data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=self.config.TEST_SIZE, random_state=self.config.RANDOM_STATE
        )

        logger.info(f"Prepared data: X_train shape {X_train.shape}, y_train shape {y_train.shape}")
        return X_train, X_test, y_train, y_test

    def train_model(self, X_train, y_train):
        logger.info("Training multi-output model...")

        self.model = MultiOutputClassifier(RandomForestClassifier(
            n_estimators=self.config.N_ESTIMATORS,
            random_state=self.config.RANDOM_STATE,
            class_weight='balanced'
        ))

        self.model.fit(X_train, y_train)
        logger.info("Model training completed")

    def evaluate_model(self, X_test, y_test):
        y_pred = self.model.predict(X_test)
        results = {}

        # Calculate feature importance for each target
        self.feature_importance = {}
        for i, (estimator, target_name) in enumerate(zip(self.model.estimators_, y_test.columns)):
            self.feature_importance[target_name] = dict(zip(
                X_test.columns,
                estimator.feature_importances_
            ))

        for i, col in enumerate(y_test.columns):
            results[col] = {
                'accuracy': accuracy_score(y_test.iloc[:, i], y_pred[:, i]),
                'precision': precision_score(y_test.iloc[:, i], y_pred[:, i], zero_division=0),
                'recall': recall_score(y_test.iloc[:, i], y_pred[:, i], zero_division=0),
                'f1': f1_score(y_test.iloc[:, i], y_pred[:, i], zero_division=0),
            }
        logger.info(f"Evaluation results: {results}")

        print("Evaluation Results:")
        for key, value in results.items():
            print(f"{key}: {value}")
        return results

def plot_feature_importance(trainer, config):
    plots_dir = os.path.join(config.OUTPUT_DIR, 'plots')
    os.makedirs(plots_dir, exist_ok=True)

    for target, importances in trainer.feature_importance.items():
        fig = px.bar(x=list(importances.keys()), y=list(importances.values()),
                    title=f'Feature Importance for {target}')
        fig.update_layout(xaxis_tickangle=-45)
        fig.write_html(os.path.join(plots_dir, f'feature_importance_{target}.html'))

def plot_performance_metrics(results, config):
    metrics_df = pd.DataFrame(results).transpose()
    plots_dir = os.path.join(config.OUTPUT_DIR, 'plots')

    for metric in ['accuracy', 'precision', 'recall', 'f1']:
        fig = px.bar(x=metrics_df.index, y=metrics_df[metric],
                    title=f'{metric.capitalize()} Scores by Target')
        fig.update_layout(xaxis_tickangle=-90)
        fig.write_html(os.path.join(plots_dir, f'performance_{metric}.html'))

    # Create a combined metrics plot
    fig = go.Figure()
    for metric in ['accuracy', 'precision', 'recall', 'f1']:
        fig.add_trace(go.Bar(name=metric, x=metrics_df.index, y=metrics_df[metric]))

    fig.update_layout(
        title='All Metrics by Target',
        barmode='group',
        xaxis_tickangle=-90
    )
    fig.write_html(os.path.join(plots_dir, 'all_metrics.html'))

    return metrics_df

def main():
    config = Config()
    preprocessor = DataPreprocessor(config)
    trainer = MultiOutputModelTrainer(config)

    logger.info("Starting enhanced anomaly detection analysis...")

    all_features, all_targets = [], []

    for filename in os.listdir(config.FOLDER_PATH):
        if filename.endswith('.xlsx'):
            file_path = os.path.join(config.FOLDER_PATH, filename)
            X, y = preprocessor.load_and_preprocess(file_path)
            if X is not None and y is not None:
                all_features.append(X)
                all_targets.append(y)

    if not all_features or not all_targets:
        logger.error("No valid data found for processing")
        return

    # Combine all features and targets
    X = pd.concat(all_features)
    y = pd.concat(all_targets)

    # Prepare and train the model
    X_train, X_test, y_train, y_test = trainer.prepare_data(X, y)
    trainer.train_model(X_train, y_train)
    results = trainer.evaluate_model(X_test, y_test)

    # Generate visualizations
    plot_feature_importance(trainer, config)
    metrics_df = plot_performance_metrics(results, config)

    # Save results
    pd.DataFrame(results).to_csv(os.path.join(config.OUTPUT_DIR, 'evaluation_results.csv'))
    pd.DataFrame(trainer.feature_importance).transpose().to_csv(
        os.path.join(config.OUTPUT_DIR, 'feature_importance.csv'))
    metrics_df.to_csv(os.path.join(config.OUTPUT_DIR, 'performance_metrics.csv'))

    logger.info("Analysis and visualizations completed successfully")

if __name__ == "__main__":
    main()

Mounted at /content/drive
Evaluation Results:
Alpha..annualisiert._w3: {'accuracy': 1.0, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
Alpha..annualisiert._w6: {'accuracy': 1.0, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
Alpha..annualisiert._w12: {'accuracy': 0.9054878048780488, 'precision': 0.8, 'recall': 0.11764705882352941, 'f1': 0.20512820512820512}
Alpha..annualisiert._w24: {'accuracy': 0.9115853658536586, 'precision': 0.6666666666666666, 'recall': 0.24242424242424243, 'f1': 0.35555555555555557}
Value.Growth_w3: {'accuracy': 1.0, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
Value.Growth_w6: {'accuracy': 0.9908536585365854, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
Value.Growth_w12: {'accuracy': 0.9390243902439024, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
Value.Growth_w24: {'accuracy': 0.9298780487804879, 'precision': 0.6, 'recall': 0.24, 'f1': 0.34285714285714286}
Small.Large_w3: {'accuracy': 1.0, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
Small.Large_w6: {'accuracy': 0.996951

In [10]:
import os
import pandas as pd
import numpy as np
import logging
from datetime import datetime
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
from google.colab import drive

try:
    drive.mount('/content/drive', force_remount=True)
except Exception as e:
    print(f"Drive mounting failed: {e}")

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger()

class Config:
    def __init__(self):
        base_path = '/content/drive/My Drive'
        self.FOLDER_PATH = os.path.join(base_path, 'Factordata')
        self.PREDICTOR_PATH = os.path.join(base_path, 'seco')
        self.OUTPUT_DIR = os.path.join(self.FOLDER_PATH, 'analysis_output')
        os.makedirs(self.OUTPUT_DIR, exist_ok=True)

        self.WINDOWS = [6, 12, 24]  # Removed 3-month window
        self.STD_THRESHOLD = 1.5    # Lowered from 2.0
        self.RANDOM_STATE = 42
        self.TEST_SIZE = 0.3
        self.N_ESTIMATORS = 100
        self.FACTOR_COLUMNS = [
            'Alpha..annualisiert.',
            'Value.Growth',
            'Small.Large',
            'Momentum',
            'Volatility'
        ]

class DataPreprocessor:
    def __init__(self, config):
        self.config = config
        self.scaler = StandardScaler()

    def load_and_preprocess(self, file_path):
        try:
            df = pd.read_excel(file_path)
            df.columns = df.columns.str.lower()

            if 'date' not in df.columns:
                raise ValueError("'date' column missing")

            df['date'] = pd.to_datetime(df['date'])
            df = df.set_index('date')

            # Standardize features
            features = []
            targets = []

            for col in self.config.FACTOR_COLUMNS:
                col_lower = col.lower()
                if col_lower not in df.columns:
                    logger.warning(f"Column {col} missing in {os.path.basename(file_path)}")
                    continue

                # Convert and clean data
                series = pd.to_numeric(df[col_lower], errors='coerce')
                series = series.ffill().bfill()  # Handle NaN values

                # Standardize the feature
                scaled_series = pd.Series(
                    self.scaler.fit_transform(series.values.reshape(-1, 1)).flatten(),
                    index=series.index
                )
                features.append(scaled_series)

                # Calculate anomalies for each window
                for window in self.config.WINDOWS:
                    rolling_mean = scaled_series.rolling(window=window).mean()
                    rolling_std = scaled_series.rolling(window=window).std()
                    outliers = ((scaled_series - rolling_mean).abs() >
                              self.config.STD_THRESHOLD * rolling_std).astype(int)
                    targets.append(outliers)

            X = pd.concat(features, axis=1)
            X.columns = self.config.FACTOR_COLUMNS

            y = pd.concat(targets, axis=1)
            y.columns = [f"{col}_w{window}" for col in self.config.FACTOR_COLUMNS
                        for window in self.config.WINDOWS]

            # Remove rows with NaN values
            valid_idx = ~(X.isna().any(axis=1) | y.isna().any(axis=1))
            X = X[valid_idx]
            y = y[valid_idx]

            return X, y

        except Exception as e:
            logger.error(f"Error processing {file_path}: {e}")
            return None, None

class AnomalyDetector:
    def __init__(self, config):
        self.config = config
        self.model = None
        self.feature_importance = {}

    def train(self, X, y):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=self.config.TEST_SIZE,
            random_state=self.config.RANDOM_STATE
        )

        self.model = MultiOutputClassifier(
            RandomForestClassifier(
                n_estimators=self.config.N_ESTIMATORS,
                random_state=self.config.RANDOM_STATE,
                class_weight='balanced'
            )
        )

        self.model.fit(X_train, y_train)
        return self.evaluate(X_test, y_test)

    def evaluate(self, X_test, y_test):
        y_pred = self.model.predict(X_test)
        results = {}

        # Calculate metrics and feature importance
        for i, (estimator, col) in enumerate(zip(self.model.estimators_, y_test.columns)):
            results[col] = {
                'accuracy': accuracy_score(y_test.iloc[:, i], y_pred[:, i]),
                'precision': precision_score(y_test.iloc[:, i], y_pred[:, i], zero_division=0),
                'recall': recall_score(y_test.iloc[:, i], y_pred[:, i], zero_division=0),
                'f1': f1_score(y_test.iloc[:, i], y_pred[:, i], zero_division=0)
            }

            self.feature_importance[col] = dict(zip(
                X_test.columns,
                estimator.feature_importances_
            ))

        return results

def save_results(results, feature_importance, config):
    # Save detailed results
    results_df = pd.DataFrame(results).transpose()
    results_df.index.name = 'target'
    results_df.to_csv(os.path.join(config.OUTPUT_DIR, 'detailed_results.csv'))

    # Save feature importance
    importance_df = pd.DataFrame(feature_importance).transpose()
    importance_df.index.name = 'target'
    importance_df.to_csv(os.path.join(config.OUTPUT_DIR, 'feature_importance.csv'))

    # Calculate and save summaries by window size
    results_df['window'] = results_df.index.str.extract(r'w(\d+)').astype(int)
    window_summary = results_df.groupby('window').mean()
    window_summary.to_csv(os.path.join(config.OUTPUT_DIR, 'window_summary.csv'))

    # Print evaluation results
    print("\nEvaluation Results:")
    for target, metrics in results.items():
        print(f"{target}: {metrics}")

def main():
    config = Config()
    preprocessor = DataPreprocessor(config)
    detector = AnomalyDetector(config)

    all_features = []
    all_targets = []

    # Process all Excel files
    for filename in os.listdir(config.FOLDER_PATH):
        if filename.endswith('.xlsx'):
            file_path = os.path.join(config.FOLDER_PATH, filename)
            X, y = preprocessor.load_and_preprocess(file_path)
            if X is not None and y is not None:
                all_features.append(X)
                all_targets.append(y)

    if not all_features:
        logger.error("No valid data found")
        return

    # Combine data and train model
    X = pd.concat(all_features)
    y = pd.concat(all_targets)

    # Train and evaluate
    results = detector.train(X, y)

    # Save results
    save_results(results, detector.feature_importance, config)

if __name__ == "__main__":
    main()

Mounted at /content/drive

Evaluation Results:
Alpha..annualisiert._w6: {'accuracy': 0.8201219512195121, 'precision': 0.5555555555555556, 'recall': 0.08333333333333333, 'f1': 0.14492753623188406}
Alpha..annualisiert._w12: {'accuracy': 0.8262195121951219, 'precision': 0.9032258064516129, 'recall': 0.34146341463414637, 'f1': 0.49557522123893805}
Alpha..annualisiert._w24: {'accuracy': 0.8414634146341463, 'precision': 0.88, 'recall': 0.4888888888888889, 'f1': 0.6285714285714286}
Value.Growth_w6: {'accuracy': 0.8201219512195121, 'precision': 0.4444444444444444, 'recall': 0.06896551724137931, 'f1': 0.11940298507462686}
Value.Growth_w12: {'accuracy': 0.8170731707317073, 'precision': 0.72, 'recall': 0.2535211267605634, 'f1': 0.375}
Value.Growth_w24: {'accuracy': 0.8658536585365854, 'precision': 0.75, 'recall': 0.43548387096774194, 'f1': 0.5510204081632653}
Small.Large_w6: {'accuracy': 0.8292682926829268, 'precision': 0.38461538461538464, 'recall': 0.09433962264150944, 'f1': 0.15151515151515152

In [11]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from google.colab import drive

try:
    drive.mount('/content/drive', force_remount=True)
except Exception as e:
    print(f"Drive mounting failed: {e}")

class Config:
    def __init__(self):
        base_path = '/content/drive/My Drive'
        self.FOLDER_PATH = os.path.join(base_path, 'Factordata')
        self.PREDICTOR_PATH = os.path.join(base_path, 'seco')
        self.OUTPUT_DIR = os.path.join(self.FOLDER_PATH, 'analysis_output')
        os.makedirs(self.OUTPUT_DIR, exist_ok=True)

        self.WINDOWS = [12, 24]
        self.STD_THRESHOLD = 1.5
        self.RANDOM_STATE = 42
        self.TEST_SIZE = 0.3
        self.FACTOR_COLUMNS = [
            'alpha..annualisiert.',
            'value.growth',
            'small.large',
            'momentum',
            'volatility'
        ]

class RatesLoader:
    def __init__(self, config):
        self.config = config

    def align_with_factor_data(self, factor_dates):
        try:
            rates_data = {}

            # Load EUR rates
            eur_path = os.path.join(self.config.PREDICTOR_PATH, 'zinskurveEUR.csv')
            if os.path.exists(eur_path):
                eur_df = pd.read_csv(eur_path, sep=';')
                if not any(eur_df.columns.str.contains('date')):
                    start_date = factor_dates.min()
                    eur_df.index = pd.date_range(start=start_date, periods=len(eur_df), freq='M')
                rates_data['eur_2y'] = pd.to_numeric(eur_df['2Y'], errors='coerce')

            # Load USD rates
            usd_path = os.path.join(self.config.PREDICTOR_PATH, 'zinskurveUSD.csv')
            if os.path.exists(usd_path):
                usd_df = pd.read_csv(usd_path, sep=';')
                if not any(usd_df.columns.str.contains('date')):
                    start_date = factor_dates.min()
                    usd_df.index = pd.date_range(start=start_date, periods=len(usd_df), freq='M')
                rates_data['usd_2y'] = pd.to_numeric(usd_df['2Y'], errors='coerce')

            # Load base rates and bonds
            base_path = os.path.join(self.config.PREDICTOR_PATH, 'geldmarktsaetze_alt.xlsx')
            bonds_path = os.path.join(self.config.PREDICTOR_PATH, 'renditen_obligationen.xlsx')

            if os.path.exists(base_path):
                base_df = pd.read_excel(base_path)
                date_col = next((col for col in base_df.columns if 'date' in col.lower()), None)
                if date_col:
                    base_df['date'] = pd.to_datetime(base_df[date_col])
                    base_df = base_df.set_index('date')
                    base_df = base_df.reindex(factor_dates, method='ffill')
                    rates_data['base_rate'] = base_df.iloc[:, 0]

            if os.path.exists(bonds_path):
                bonds_df = pd.read_excel(bonds_path)
                date_col = next((col for col in bonds_df.columns if 'date' in col.lower()), None)
                if date_col:
                    bonds_df['date'] = pd.to_datetime(bonds_df[date_col])
                    bonds_df = bonds_df.set_index('date')
                    bonds_df = bonds_df.reindex(factor_dates, method='ffill')
                    rates_data['bond_2y'] = bonds_df['2Y'] if '2Y' in bonds_df.columns else bonds_df.iloc[:, 1]

            rates_df = pd.DataFrame(rates_data)
            rates_df = rates_df.reindex(factor_dates)
            print(f"Loaded rates with shape: {rates_df.shape}")
            print(f"Date range: {rates_df.index.min()} to {rates_df.index.max()}")
            return rates_df

        except Exception as e:
            print(f"Error loading rates: {e}")
            return None

class SecoFeatureLoader:
    def __init__(self, config):
        self.config = config

    def load_features(self):
        try:
            hist_df = self._load_seco_file('ks_q_hist.csv')
            current_df = self._load_seco_file('ks_q 2.csv')

            combined = pd.concat([hist_df, current_df])
            combined = combined[~combined.index.duplicated(keep='last')]
            combined = combined.sort_index()
            monthly = combined.resample('M').ffill()

            return monthly

        except Exception as e:
            print(f"Error loading SECO features: {e}")
            return None

    def _load_seco_file(self, filename):
        file_path = os.path.join(self.config.PREDICTOR_PATH, filename)
        df = pd.read_csv(file_path)
        df['date'] = pd.to_datetime(df['date'])

        df = df.pivot_table(
            index='date',
            columns=['structure', 'type'],
            values='value',
            aggfunc='first'
        )

        df.columns = [f'seco_{str(col[0]).replace(" ", "_")}_{str(col[1]).replace(" ", "_")}'
                     for col in df.columns]
        return df

class DataProcessor:
    def __init__(self, config):
        self.config = config
        self.scaler = StandardScaler()
        self.seco_loader = SecoFeatureLoader(config)

    def prepare_data(self):
        # Load and preprocess factor data
        factor_data = self._load_factor_data()
        if factor_data is None:
            return None, None

        # Load additional features
        rates_loader = RatesLoader(self.config)
        rates_data = rates_loader.align_with_factor_data(factor_data.index)
        seco_features = self.seco_loader.load_features()

        # Combine features
        features = [factor_data]
        if seco_features is not None:
            features.append(seco_features)
        if rates_data is not None:
            features.append(rates_data)

        # Align dates and combine
        common_dates = features[0].index
        for df in features[1:]:
            common_dates = common_dates.intersection(df.index)

        X = pd.concat([df.loc[common_dates] for df in features], axis=1)

        # Calculate targets
        y = self._calculate_anomalies(factor_data)

        # Clean data
        valid_idx = ~(X.isna().any(axis=1) | y.isna().any(axis=1))
        X = X.loc[valid_idx]
        y = y.loc[valid_idx]

        print(f"\nFinal data shapes - X: {X.shape}, y: {y.shape}")
        print(f"Date range: {X.index.min()} to {X.index.max()}")
        print(f"Features included: {X.columns.tolist()}")

        return X, y

    def _load_factor_data(self):
        for filename in os.listdir(self.config.FOLDER_PATH):
            if filename.endswith('.xlsx'):
                print(f"Processing file: {filename}")
                file_path = os.path.join(self.config.FOLDER_PATH, filename)
                df = pd.read_excel(file_path)
                print(f"Original columns: {df.columns.tolist()}")

                # Process each factor column individually
                processed_data = {}
                for col in self.config.FACTOR_COLUMNS:
                    # Find the matching column ignoring case
                    orig_col = next((c for c in df.columns if c.lower() == col), None)
                    if orig_col:
                        processed_data[col] = self.scaler.fit_transform(df[[orig_col]]).flatten()

                # Create new dataframe with processed columns
                df_processed = pd.DataFrame(processed_data, index=pd.to_datetime(df['Date']))
                print(f"Processed columns: {df_processed.columns.tolist()}")
                return df_processed

        return None

    def _calculate_anomalies(self, data):
        anomalies = []
        for col in data.columns:
            series = data[col]
            for window in self.config.WINDOWS:
                rolling_mean = series.rolling(window=window).mean()
                rolling_std = series.rolling(window=window).std()
                outliers = ((series - rolling_mean).abs() >
                          self.config.STD_THRESHOLD * rolling_std).astype(int)
                outliers.name = f"{col}_w{window}"
                anomalies.append(outliers)

        return pd.concat(anomalies, axis=1)

class EnhancedLearningModel:
    def __init__(self, config):
        self.config = config
        self.rf_model = MultiOutputClassifier(
            RandomForestClassifier(n_estimators=100,
                                 class_weight='balanced',
                                 random_state=config.RANDOM_STATE))
        self.gb_model = MultiOutputClassifier(
            GradientBoostingClassifier(n_estimators=100,
                                     random_state=config.RANDOM_STATE))

    def train(self, X, y):
        self.rf_model.fit(X, y)
        self.gb_model.fit(X, y)

    def predict(self, X):
        rf_pred = self.rf_model.predict_proba(X)
        gb_pred = self.gb_model.predict_proba(X)

        final_pred = []
        for rf_p, gb_p in zip(rf_pred, gb_pred):
            ensemble_p = np.mean([rf_p, gb_p], axis=0)
            final_pred.append(ensemble_p.argmax(axis=1))

        return np.column_stack(final_pred)

    def get_feature_importance(self):
        importance_dict = {}
        for i, estimator in enumerate(self.rf_model.estimators_):
            importance_dict[f'target_{i}'] = dict(zip(
                range(len(estimator.feature_importances_)),
                estimator.feature_importances_
            ))
        return importance_dict

def evaluate_predictions(y_true, y_pred):
    results = {}
    for i, col in enumerate(y_true.columns):
        results[col] = {
            'accuracy': accuracy_score(y_true.iloc[:, i], y_pred[:, i]),
            'precision': precision_score(y_true.iloc[:, i], y_pred[:, i], zero_division=0),
            'recall': recall_score(y_true.iloc[:, i], y_pred[:, i], zero_division=0),
            'f1': f1_score(y_true.iloc[:, i], y_pred[:, i], zero_division=0)
        }
    return results

def main():
    config = Config()
    processor = DataProcessor(config)

    X, y = processor.prepare_data()
    if X is None or y is None:
        print("Failed to prepare data")
        return

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=config.TEST_SIZE,
        random_state=config.RANDOM_STATE
    )

    model = EnhancedLearningModel(config)
    model.train(X_train, y_train)

    y_pred = model.predict(X_test)
    results = evaluate_predictions(y_test, y_pred)

    print("\nEvaluation Results:")
    for target, metrics in results.items():
        print(f"{target}: {metrics}")

    importance = model.get_feature_importance()
    pd.DataFrame(importance).to_csv(
        os.path.join(config.OUTPUT_DIR, 'feature_importance.csv'))

if __name__ == "__main__":
    main()

Mounted at /content/drive
Processing file: Pictet.xlsx
Original columns: ['Date', 'Alpha..annualisiert.', 'Value.Growth', 'Small.Large', 'Momentum', 'Volatility']
Processed columns: ['alpha..annualisiert.', 'value.growth', 'small.large', 'momentum', 'volatility']


<ipython-input-11-d70a824c3ac8>:79: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



Error loading rates: Unknown datetime string format, unable to parse: https://data.snb.ch, at position 0
Error loading SECO features: [Errno 2] No such file or directory: '/content/drive/My Drive/seco/ks_q 2.csv'

Final data shapes - X: (91, 5), y: (91, 10)
Date range: 2016-09-30 12:00:00 to 2024-03-31 12:00:00
Features included: ['alpha..annualisiert.', 'value.growth', 'small.large', 'momentum', 'volatility']

Evaluation Results:
alpha..annualisiert._w12: {'accuracy': 0.6785714285714286, 'precision': 0.75, 'recall': 0.46153846153846156, 'f1': 0.5714285714285714}
alpha..annualisiert._w24: {'accuracy': 0.8571428571428571, 'precision': 0.8571428571428571, 'recall': 0.6666666666666666, 'f1': 0.75}
value.growth_w12: {'accuracy': 0.8928571428571429, 'precision': 0.5, 'recall': 0.3333333333333333, 'f1': 0.4}
value.growth_w24: {'accuracy': 0.9642857142857143, 'precision': 1.0, 'recall': 0.6666666666666666, 'f1': 0.8}
small.large_w12: {'accuracy': 0.75, 'precision': 0.5, 'recall': 0.1428571428

In [12]:
class RatesLoader:
    def __init__(self, config):
        self.config = config

    def align_with_factor_data(self, factor_dates):
        try:
            rates_data = {}

            # Load EUR rates
            eur_path = os.path.join(self.config.PREDICTOR_PATH, 'zinskurveEUR.csv')
            if os.path.exists(eur_path):
                eur_df = pd.read_csv(eur_path, sep=';')
                if not any(eur_df.columns.str.contains('date')):
                    start_date = factor_dates.min()
                    eur_df.index = pd.date_range(start=start_date, periods=len(eur_df), freq='M')
                rates_data['eur_2y'] = pd.to_numeric(eur_df['2Y'], errors='coerce')

            # Load USD rates
            usd_path = os.path.join(self.config.PREDICTOR_PATH, 'zinskurveUSD.csv')
            if os.path.exists(usd_path):
                usd_df = pd.read_csv(usd_path, sep=';')
                if not any(usd_df.columns.str.contains('date')):
                    start_date = factor_dates.min()
                    usd_df.index = pd.date_range(start=start_date, periods=len(usd_df), freq='M')
                rates_data['usd_2y'] = pd.to_numeric(usd_df['2Y'], errors='coerce')

            # Load base rates and bonds
            base_path = os.path.join(self.config.PREDICTOR_PATH, 'geldmarktsaetze_alt.xlsx')
            bonds_path = os.path.join(self.config.PREDICTOR_PATH, 'renditen_obligationen.xlsx')

            if os.path.exists(base_path):
                # Debug print for base rates loading
                print("Loading base rates file...")
                # First check the structure
                try:
                    # Read with header inspection
                    header_df = pd.read_excel(base_path, nrows=5)
                    print("First 5 rows of raw data:")
                    print(header_df)

                    # Try reading with skipped rows
                    base_df = pd.read_excel(base_path, skiprows=3)
                    print("\nAfter skipping header rows:")
                    print(base_df.head())

                    # Find date column
                    date_col = next((col for col in base_df.columns if 'date' in col.lower()), None)
                    if date_col:
                        print(f"\nFound date column: {date_col}")
                        print("Sample dates before parsing:", base_df[date_col].head())

                        # Try date parsing with error handling
                        base_df['date'] = pd.to_datetime(base_df[date_col], errors='coerce')
                        # Remove rows where date parsing failed
                        base_df = base_df.dropna(subset=['date'])

                        print("Sample dates after parsing:", base_df['date'].head())

                        if not base_df.empty:
                            base_df = base_df.set_index('date')
                            base_df = base_df.reindex(factor_dates, method='ffill')
                            rates_data['base_rate'] = base_df.iloc[:, 0]
                        else:
                            print("No valid dates found after parsing")
                    else:
                        print("Available columns:", base_df.columns.tolist())
                        print("No date column found")

                except Exception as e:
                    print(f"Error processing base rates: {e}")

            if os.path.exists(bonds_path):
                bonds_df = pd.read_excel(bonds_path)
                date_col = next((col for col in bonds_df.columns if 'date' in col.lower()), None)
                if date_col:
                    bonds_df['date'] = pd.to_datetime(bonds_df[date_col])
                    bonds_df = bonds_df.set_index('date')
                    bonds_df = bonds_df.reindex(factor_dates, method='ffill')
                    rates_data['bond_2y'] = bonds_df['2Y'] if '2Y' in bonds_df.columns else bonds_df.iloc[:, 1]

            rates_df = pd.DataFrame(rates_data)
            rates_df = rates_df.reindex(factor_dates)
            print(f"Loaded rates with shape: {rates_df.shape}")
            print(f"Date range: {rates_df.index.min()} to {rates_df.index.max()}")
            return rates_df

        except Exception as e:
            print(f"Error loading rates: {e}")
            import traceback
            print(f"Full traceback:", traceback.format_exc())
            return None

In [13]:
class RatesLoader:
    def __init__(self, config):
        self.config = config

    def align_with_factor_data(self, factor_dates):
        try:
            rates_data = {}

            # Load EUR rates
            eur_path = os.path.join(self.config.PREDICTOR_PATH, 'zinskurveEUR.csv')
            if os.path.exists(eur_path):
                eur_df = pd.read_csv(eur_path, sep=';')
                if not any(eur_df.columns.str.contains('date')):
                    start_date = factor_dates.min()
                    eur_df.index = pd.date_range(start=start_date, periods=len(eur_df), freq='M')
                rates_data['eur_2y'] = pd.to_numeric(eur_df['2Y'], errors='coerce')

            # Load USD rates
            usd_path = os.path.join(self.config.PREDICTOR_PATH, 'zinskurveUSD.csv')
            if os.path.exists(usd_path):
                usd_df = pd.read_csv(usd_path, sep=';')
                if not any(usd_df.columns.str.contains('date')):
                    start_date = factor_dates.min()
                    usd_df.index = pd.date_range(start=start_date, periods=len(usd_df), freq='M')
                rates_data['usd_2y'] = pd.to_numeric(usd_df['2Y'], errors='coerce')

            # Load base rates with improved handling
            base_path = os.path.join(self.config.PREDICTOR_PATH, 'geldmarktsaetze_alt.xlsx')
            bonds_path = os.path.join(self.config.PREDICTOR_PATH, 'renditen_obligationen.xlsx')

            if os.path.exists(base_path):
                print("Loading base rates file...")

                # Try different numbers of rows to skip until we find the actual data
                for skip_rows in range(10):
                    try:
                        base_df = pd.read_excel(base_path, skiprows=skip_rows)
                        print(f"\nTrying with skiprows={skip_rows}:")
                        print(base_df.head())

                        # Check if this looks like the actual data (has dates and numbers)
                        first_col = base_df.iloc[:, 0]
                        if pd.to_datetime(first_col, errors='coerce').notna().any():
                            print(f"Found data starting at row {skip_rows}")

                            # Clean up the dataframe
                            base_df = base_df.dropna(how='all')  # Remove completely empty rows
                            base_df = base_df.dropna(axis=1, how='all')  # Remove empty columns

                            # Convert first column to datetime
                            base_df['date'] = pd.to_datetime(base_df.iloc[:, 0], errors='coerce')
                            base_df = base_df.dropna(subset=['date'])

                            if not base_df.empty:
                                print("Successfully loaded data:")
                                print(base_df.head())

                                base_df = base_df.set_index('date')
                                base_df = base_df.reindex(factor_dates, method='ffill')
                                rates_data['base_rate'] = base_df.iloc[:, 0]
                                break
                    except Exception as e:
                        print(f"Error with skiprows={skip_rows}: {e}")
                        continue

            if os.path.exists(bonds_path):
                bonds_df = pd.read_excel(bonds_path)
                date_col = next((col for col in bonds_df.columns if 'date' in col.lower()), None)
                if date_col:
                    bonds_df['date'] = pd.to_datetime(bonds_df[date_col])
                    bonds_df = bonds_df.set_index('date')
                    bonds_df = bonds_df.reindex(factor_dates, method='ffill')
                    rates_data['bond_2y'] = bonds_df['2Y'] if '2Y' in bonds_df.columns else bonds_df.iloc[:, 1]

            rates_df = pd.DataFrame(rates_data)
            rates_df = rates_df.reindex(factor_dates)
            print(f"\nFinal loaded rates shape: {rates_df.shape}")
            print(f"Date range: {rates_df.index.min()} to {rates_df.index.max()}")
            return rates_df

        except Exception as e:
            print(f"Error loading rates: {e}")
            import traceback
            print(f"Full traceback:", traceback.format_exc())
            return None

In [14]:
import os
import pandas as pd
import numpy as np
import traceback
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from google.colab import drive

# Create an instance of the configuration
config = Config()

# Create and use the RatesLoader
factor_dates = pd.date_range(start='2016-09-30', end='2024-03-31', freq='M')
loader = RatesLoader(config)
rates_data = loader.align_with_factor_data(factor_dates)

# Print the results
if rates_data is not None:
    print("\nSuccessfully loaded rates data:")
    print(rates_data.head())
    print("\nShape:", rates_data.shape)
    print("Date range:", rates_data.index.min(), "to", rates_data.index.max())
else:
    print("\nFailed to load rates data")

# Continue with the rest of your analysis
# Execute the main function
if __name__ == "__main__":
    main()

<ipython-input-14-aeece0cdcf7f>:16: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.

<ipython-input-13-f82569f064fa>:70: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



Error loading rates: Unknown datetime string format, unable to parse: https://data.snb.ch, at position 0
Full traceback: Traceback (most recent call last):
  File "<ipython-input-13-f82569f064fa>", line 70, in align_with_factor_data
    bonds_df['date'] = pd.to_datetime(bonds_df[date_col])
  File "/usr/local/lib/python3.10/dist-packages/pandas/core/tools/datetimes.py", line 1067, in to_datetime
    values = convert_listlike(arg._values, format)
  File "/usr/local/lib/python3.10/dist-packages/pandas/core/tools/datetimes.py", line 435, in _convert_listlike_datetimes
    result, tz_parsed = objects_to_datetime64(
  File "/usr/local/lib/python3.10/dist-packages/pandas/core/arrays/datetimes.py", line 2398, in objects_to_datetime64
    result, tz_parsed = tslib.array_to_datetime(
  File "tslib.pyx", line 414, in pandas._libs.tslib.array_to_datetime
  File "tslib.pyx", line 596, in pandas._libs.tslib.array_to_datetime
  File "tslib.pyx", line 553, in pandas._libs.tslib.array_to_datetime
  Fil

<ipython-input-13-f82569f064fa>:70: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



Error loading rates: Unknown datetime string format, unable to parse: https://data.snb.ch, at position 0
Full traceback: Traceback (most recent call last):
  File "<ipython-input-13-f82569f064fa>", line 70, in align_with_factor_data
    bonds_df['date'] = pd.to_datetime(bonds_df[date_col])
  File "/usr/local/lib/python3.10/dist-packages/pandas/core/tools/datetimes.py", line 1067, in to_datetime
    values = convert_listlike(arg._values, format)
  File "/usr/local/lib/python3.10/dist-packages/pandas/core/tools/datetimes.py", line 435, in _convert_listlike_datetimes
    result, tz_parsed = objects_to_datetime64(
  File "/usr/local/lib/python3.10/dist-packages/pandas/core/arrays/datetimes.py", line 2398, in objects_to_datetime64
    result, tz_parsed = tslib.array_to_datetime(
  File "tslib.pyx", line 414, in pandas._libs.tslib.array_to_datetime
  File "tslib.pyx", line 596, in pandas._libs.tslib.array_to_datetime
  File "tslib.pyx", line 553, in pandas._libs.tslib.array_to_datetime
  Fil

In [19]:
class RatesLoader:
    def __init__(self, config):
        self.config = config

    def align_with_factor_data(self, factor_dates):
        try:
            rates_data = {}

            # Load and process geldmarktsaetze.xlsx
            base_path = os.path.join(self.config.PREDICTOR_PATH, 'geldmarktsaetze.xlsx')

            if os.path.exists(base_path):
                print("Loading interest rates file...")

                # Read Excel file with explicit dtype
                rates_df = pd.read_excel(base_path, header=0, dtype={'date': str})

                # Take first three columns if available
                if len(rates_df.columns) >= 3:
                    rates_df = rates_df.iloc[:, :3]

                # Convert first column to datetime and set as index
                rates_df.iloc[:, 0] = pd.to_datetime(rates_df.iloc[:, 0], format='%Y-%m')
                rates_df.set_index(rates_df.columns[0], inplace=True)

                # Take the 2Y rates (last column)
                rates_series = rates_df.iloc[:, -1].astype(float)

                # Resample to month-end using ME
                rates_monthly = rates_series.resample('ME').last()

                # Align with factor dates using forward fill
                aligned_rates = rates_monthly.reindex(factor_dates, method='ffill')
                rates_data['rates_2y'] = aligned_rates

                print(f"Loaded rates data from {rates_series.index.min()} to {rates_series.index.max()}")

            # Load EUR rates if available
            eur_path = os.path.join(self.config.PREDICTOR_PATH, 'zinskurveEUR.csv')
            if os.path.exists(eur_path):
                eur_df = pd.read_csv(eur_path, sep=';')
                if not any(eur_df.columns.str.contains('date')):
                    start_date = factor_dates.min()
                    eur_df.index = pd.date_range(start=start_date, periods=len(eur_df), freq='ME')
                rates_data['eur_2y'] = pd.to_numeric(eur_df['2Y'], errors='coerce')

            # Load USD rates if available
            usd_path = os.path.join(self.config.PREDICTOR_PATH, 'zinskurveUSD.csv')
            if os.path.exists(usd_path):
                usd_df = pd.read_csv(usd_path, sep=';')
                if not any(usd_df.columns.str.contains('date')):
                    start_date = factor_dates.min()
                    usd_df.index = pd.date_range(start=start_date, periods=len(usd_df), freq='ME')
                rates_data['usd_2y'] = pd.to_numeric(usd_df['2Y'], errors='coerce')

            # Create final DataFrame with explicit dtype
            rates_df = pd.DataFrame(rates_data, dtype=float)
            print(f"\nFinal loaded rates shape: {rates_df.shape}")
            print(f"Date range: {rates_df.index.min()} to {rates_df.index.max()}")
            return rates_df

        except Exception as e:
            print(f"Error loading rates: {e}")
            import traceback
            print(f"Full traceback:", traceback.format_exc())
            return None

In [23]:
class RatesLoader:
    def __init__(self, config):
        self.config = config

    def align_with_factor_data(self, factor_dates):
        try:
            rates_data = {}

            # Load and process geldmarktsaetze.xlsx
            base_path = os.path.join(self.config.PREDICTOR_PATH, 'geldmarktsaetze.xlsx')

            if os.path.exists(base_path):
                print("Loading interest rates file...")

                # Read Excel file
                rates_df = pd.read_excel(base_path, header=0)

                print("First few rows of raw data:")
                print(rates_df.head())

                # Convert first column to datetime with more flexible parsing
                try:
                    # First try direct parsing
                    rates_df.iloc[:, 0] = pd.to_datetime(rates_df.iloc[:, 0], format='%Y-%m')
                except:
                    # If that fails, try manual string manipulation
                    dates = rates_df.iloc[:, 0].apply(lambda x: f"{x}-01" if isinstance(x, str) else x)
                    rates_df.iloc[:, 0] = pd.to_datetime(dates)

                print("\nAfter date conversion:")
                print(rates_df.head())

                # Set first column as index
                rates_df.set_index(rates_df.columns[0], inplace=True)

                # Take the 2Y rates (last column)
                rates_series = rates_df.iloc[:, -1].astype(float)

                # Resample to month-end using ME
                rates_monthly = rates_series.resample('ME').last()

                # Align with factor dates using forward fill
                aligned_rates = rates_monthly.reindex(factor_dates, method='ffill')
                rates_data['rates_2y'] = aligned_rates

                print(f"\nLoaded rates data from {rates_series.index.min()} to {rates_series.index.max()}")
                print("\nSample of aligned rates data:")
                print(aligned_rates.head())

            # Create final DataFrame with explicit dtype
            rates_df = pd.DataFrame(rates_data, dtype=float)
            print(f"\nFinal loaded rates shape: {rates_df.shape}")
            print(f"Date range: {rates_df.index.min()} to {rates_df.index.max()}")
            return rates_df

        except Exception as e:
            print(f"Error loading rates: {e}")
            import traceback
            print(f"Full traceback:", traceback.format_exc())
            return None

In [24]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from google.colab import drive
import traceback

try:
    drive.mount('/content/drive', force_remount=True)
except Exception as e:
    print(f"Drive mounting failed: {e}")

print("\nFirst testing RatesLoader independently...")
config = Config()
factor_dates = pd.date_range(start='2016-09-30', end='2024-03-31', freq='ME')
loader = RatesLoader(config)
test_rates = loader.align_with_factor_data(factor_dates)

if test_rates is not None:
    print("\nRates loader test successful!")
    print("Now running full model...\n")
else:
    print("\nRates loader test failed! Check the errors above.")
    print("Continuing with full model anyway...\n")

print("\nStarting full model execution...")
results, importance = main()
print("\nExecution completed!")

Mounted at /content/drive

First testing RatesLoader independently...
Loading interest rates file...
First few rows of raw data:
      date  EPB@SNB.rendoblim{1J}  EPB@SNB.rendoblim{2J}  \
0  1988-01                  2.887                  3.218   
1  1988-02                  2.638                  2.990   
2  1988-03                  2.641                  3.263   
3  1988-04                  2.800                  3.250   
4  1988-05                  3.191                  3.602   

   EPB@SNB.rendoblim{3J}  EPB@SNB.rendoblim{4J}  EPB@SNB.rendoblim{5J}  \
0                  3.393                  3.554                  3.695   
1                  3.271                  3.498                  3.672   
2                  3.573                  3.734                  3.823   
3                  3.467                  3.594                  3.684   
4                  3.802                  3.909                  3.974   

   EPB@SNB.rendoblim{6J}  EPB@SNB.rendoblim{7J}  EPB@SNB.rendobli

/usr/local/lib/python3.10/dist-packages/pandas/core/indexes/base.py:7588: FutureWarning:

Dtype inference on a pandas object (Series, Index, ExtensionArray) is deprecated. The Index constructor will keep the original dtype in the future. Call `infer_objects` on the result to get the old behavior.



First few rows of raw data:
      date  EPB@SNB.rendoblim{1J}  EPB@SNB.rendoblim{2J}  \
0  1988-01                  2.887                  3.218   
1  1988-02                  2.638                  2.990   
2  1988-03                  2.641                  3.263   
3  1988-04                  2.800                  3.250   
4  1988-05                  3.191                  3.602   

   EPB@SNB.rendoblim{3J}  EPB@SNB.rendoblim{4J}  EPB@SNB.rendoblim{5J}  \
0                  3.393                  3.554                  3.695   
1                  3.271                  3.498                  3.672   
2                  3.573                  3.734                  3.823   
3                  3.467                  3.594                  3.684   
4                  3.802                  3.909                  3.974   

   EPB@SNB.rendoblim{6J}  EPB@SNB.rendoblim{7J}  EPB@SNB.rendoblim{8J}  \
0                  3.810                  3.904                  3.980   
1                 

/usr/local/lib/python3.10/dist-packages/pandas/core/indexes/base.py:7588: FutureWarning:

Dtype inference on a pandas object (Series, Index, ExtensionArray) is deprecated. The Index constructor will keep the original dtype in the future. Call `infer_objects` on the result to get the old behavior.




Making predictions...

Evaluation Results:
alpha..annualisiert._w12: {'accuracy': 0.6071428571428571, 'precision': 0.6666666666666666, 'recall': 0.3076923076923077, 'f1': 0.42105263157894735}
alpha..annualisiert._w24: {'accuracy': 0.8571428571428571, 'precision': 0.8571428571428571, 'recall': 0.6666666666666666, 'f1': 0.75}
value.growth_w12: {'accuracy': 0.8928571428571429, 'precision': 0.5, 'recall': 0.3333333333333333, 'f1': 0.4}
value.growth_w24: {'accuracy': 0.9642857142857143, 'precision': 1.0, 'recall': 0.6666666666666666, 'f1': 0.8}
small.large_w12: {'accuracy': 0.7857142857142857, 'precision': 1.0, 'recall': 0.14285714285714285, 'f1': 0.25}
small.large_w24: {'accuracy': 0.9285714285714286, 'precision': 1.0, 'recall': 0.3333333333333333, 'f1': 0.5}
momentum_w12: {'accuracy': 0.7857142857142857, 'precision': 0.5, 'recall': 0.16666666666666666, 'f1': 0.25}
momentum_w24: {'accuracy': 0.8571428571428571, 'precision': 0.75, 'recall': 0.5, 'f1': 0.6}
volatility_w12: {'accuracy': 0.60

In [25]:
def main():
    config = Config()
    processor = DataProcessor(config)

    X, y = processor.prepare_data()
    if X is None or y is None:
        print("Failed to prepare data")
        return

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=config.TEST_SIZE,
        random_state=config.RANDOM_STATE
    )

    model = EnhancedLearningModel(config)
    model.train(X_train, y_train)

    y_pred = model.predict(X_test)
    results = evaluate_predictions(y_test, y_pred)

    print("\nEvaluation Results:")
    for target, metrics in results.items():
        print(f"{target}: {metrics}")

    # Convert results to DataFrame for visualization
    df = pd.DataFrame.from_dict(results, orient='index')
    df['factor'] = df.index.str.split('_w').str[0]
    df['window'] = df.index.str.split('_w').str[1].astype(int)

    # Create visualization
    plt.style.use('seaborn')
    fig, axes = plt.subplots(2, 2, figsize=(20, 15))
    fig.suptitle('Model Performance Comparison: 12-Month vs 24-Month Windows', fontsize=16, y=0.95)

    # Colors for windows
    colors = ['#3498db', '#2ecc71']  # Blue for 12m, Green for 24m

    # Metrics to plot
    metrics = ['accuracy', 'precision', 'recall', 'f1']
    titles = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
    factors = df['factor'].unique()
    x = np.arange(len(factors))
    width = 0.35

    # Create all four plots
    for idx, (metric, title) in enumerate(zip(metrics, titles)):
        row = idx // 2
        col = idx % 2
        ax = axes[row, col]

        ax.bar(x - width/2, df[df['window']==12][metric], width, label='12-Month', color=colors[0])
        ax.bar(x + width/2, df[df['window']==24][metric], width, label='24-Month', color=colors[1])
        ax.set_xticks(x)
        ax.set_xticklabels(factors, rotation=45)
        ax.set_title(f'{title} by Factor and Window')
        ax.set_ylabel(title)
        ax.grid(True, alpha=0.3)
        ax.legend()

    plt.tight_layout()
    plt.show()

    # Print summary statistics
    print("\nAverage Performance by Window:")
    print(df.groupby('window')[metrics].mean().round(3))

    print("\nBest Performing Factors:")
    for metric in metrics:
        best_idx = df[metric].idxmax()
        print(f"\nBest {metric.upper()}:")
        print(f"Factor: {df.loc[best_idx, 'factor']}")
        print(f"Window: {df.loc[best_idx, 'window']} months")
        print(f"Score: {df.loc[best_idx, metric]:.3f}")

    # Save feature importance
    importance = model.get_feature_importance()
    pd.DataFrame(importance).to_csv(
        os.path.join(config.OUTPUT_DIR, 'feature_importance.csv'))

    return results, importance

if __name__ == "__main__":
    print("Starting model execution...\n")
    results, importance = main()
    print("\nExecution completed!")

Starting model execution...

Processing file: Pictet.xlsx
Original columns: ['Date', 'Alpha..annualisiert.', 'Value.Growth', 'Small.Large', 'Momentum', 'Volatility']
Processed columns: ['alpha..annualisiert.', 'value.growth', 'small.large', 'momentum', 'volatility']
Loading interest rates file...
First few rows of raw data:
      date  EPB@SNB.rendoblim{1J}  EPB@SNB.rendoblim{2J}  \
0  1988-01                  2.887                  3.218   
1  1988-02                  2.638                  2.990   
2  1988-03                  2.641                  3.263   
3  1988-04                  2.800                  3.250   
4  1988-05                  3.191                  3.602   

   EPB@SNB.rendoblim{3J}  EPB@SNB.rendoblim{4J}  EPB@SNB.rendoblim{5J}  \
0                  3.393                  3.554                  3.695   
1                  3.271                  3.498                  3.672   
2                  3.573                  3.734                  3.823   
3                

/usr/local/lib/python3.10/dist-packages/pandas/core/indexes/base.py:7588: FutureWarning:

Dtype inference on a pandas object (Series, Index, ExtensionArray) is deprecated. The Index constructor will keep the original dtype in the future. Call `infer_objects` on the result to get the old behavior.



Error loading SECO features: [Errno 2] No such file or directory: '/content/drive/My Drive/seco/ks_q 2.csv'

Final data shapes - X: (91, 6), y: (91, 10)
Date range: 2016-09-30 12:00:00 to 2024-03-31 12:00:00
Features included: ['alpha..annualisiert.', 'value.growth', 'small.large', 'momentum', 'volatility', 'rates_2y']

Evaluation Results:
alpha..annualisiert._w12: {'accuracy': 0.6071428571428571, 'precision': 0.6666666666666666, 'recall': 0.3076923076923077, 'f1': 0.42105263157894735}
alpha..annualisiert._w24: {'accuracy': 0.8571428571428571, 'precision': 0.8571428571428571, 'recall': 0.6666666666666666, 'f1': 0.75}
value.growth_w12: {'accuracy': 0.8928571428571429, 'precision': 0.5, 'recall': 0.3333333333333333, 'f1': 0.4}
value.growth_w24: {'accuracy': 0.9642857142857143, 'precision': 1.0, 'recall': 0.6666666666666666, 'f1': 0.8}
small.large_w12: {'accuracy': 0.7857142857142857, 'precision': 1.0, 'recall': 0.14285714285714285, 'f1': 0.25}
small.large_w24: {'accuracy': 0.928571428571

NameError: name 'plt' is not defined